# Notebook 06 — Hypothesis C: AggBond silhouette peaks and macro episodes

**Hypothesis:** High **silhouette** on **AggBond** coincides with **bond-market stress episodes** (European debt 2012–13, China/oil 2015–16, repo/COVID 2019–20). Under H0, silhouette is i.i.d. across refits; under H1, labelled **crisis windows** show higher silhouette.

**Tests:** **Mann–Whitney** and **Welch t-test** on silhouette (crisis vs other). **OLS + HAC:** silhouette ~ episode dummies + time trend + optional VIX-cycle control.


In [ ]:
import sys
import pickle
import warnings
from pathlib import Path

_here = Path.cwd().resolve()
_repo_root = _here
for _ in range(16):
    if (_repo_root / "src" / "config" / "settings.py").is_file():
        break
    if _repo_root.parent == _repo_root:
        raise RuntimeError("Cannot find repo root")
    _repo_root = _repo_root.parent
sys.path.insert(0, str(_repo_root))
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from scipy import stats

from src.utils.helpers import setup_logging
setup_logging()

RESULTS_DIR = _repo_root / "results"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

from src.config.settings import (
    ASSETS, FRED_SERIES, ASSET_TICKERS,
    DATA_START, DATA_END, TRAIN_YEARS,
)
from src.data.loader import DataLoader
from src.data.preprocessor import DataPreprocessor
from src.cluster_stability import build_internal_validity_df
from src.analysis.time_series_stats import macro_cycle_at_date, hac_ols

STABILITY_CACHE = RESULTS_DIR / "stability_jm_fits.pkl"
if not STABILITY_CACHE.exists():
    raise FileNotFoundError("Run notebook 03 first to create stability_jm_fits.pkl")
with open(STABILITY_CACHE, "rb") as f:
    jm_fits = pickle.load(f)

cvi_df = build_internal_validity_df(jm_fits, ASSETS)
cvi_df = cvi_df[(cvi_df["Asset"] == "AggBond") & cvi_df["has_features"]].copy()

loader = DataLoader()
prices = loader.load_prices(ASSET_TICKERS, start=DATA_START, end=DATA_END)
fred = loader.load_fred(FRED_SERIES, start=DATA_START, end=DATA_END)
prep = DataPreprocessor()
excess_returns, _, fred_aligned = prep.prepare(prices, fred)
vix = fred_aligned["vix"].astype(float)

# Narrative windows (inclusive on calendar; refit dates inside get flag 1 if any gap)
EPISODES = [
    ("Euro debt / whatever-it-takes", "2012-01-01", "2013-12-31"),
    ("China slowdown / oil", "2015-01-01", "2016-12-31"),
    ("Repo / COVID", "2019-09-01", "2020-12-31"),
]

def in_any_episode(ts):
    t = pd.Timestamp(ts)
    for _, a, b in EPISODES:
        if pd.Timestamp(a) <= t <= pd.Timestamp(b):
            return 1
    return 0

cvi_df["crisis_episode"] = cvi_df["Rebal_date"].map(in_any_episode)
cvi_df["vix_cycle"] = [macro_cycle_at_date(vix, d) for d in cvi_df["Rebal_date"]]
cvi_df["trend_time"] = (cvi_df["Rebal_date"] - cvi_df["Rebal_date"].min()).dt.days / 365.25
print(cvi_df.head())


In [ ]:
y_c = cvi_df.loc[cvi_df["crisis_episode"] == 1, "silhouette"].dropna()
y_o = cvi_df.loc[cvi_df["crisis_episode"] == 0, "silhouette"].dropna()
if len(y_c) > 0 and len(y_o) > 0:
    mw = stats.mannwhitneyu(y_c, y_o, alternative="two-sided")
    mwp = float(mw.pvalue)
else:
    mwp = float("nan")
try:
    if len(y_c) > 1 and len(y_o) > 1:
        tt = stats.ttest_ind(y_c, y_o, equal_var=False)
        ttp = float(tt.pvalue)
    else:
        ttp = float("nan")
except Exception:
    ttp = float("nan")
print("Mann–Whitney U p-value:", round(mwp, 4))
print("Welch t-test p-value:", round(ttp, 4))
print("crisis n", len(y_c), "other n", len(y_o), "mean crisis", y_c.mean(), "mean other", y_o.mean())
with open(RESULTS_DIR / "hypothesis_C_tests.txt", "w", encoding="utf-8") as f:
    f.write(f"mannwhitney_p={mwp}\nwelch_p={ttp}\n")


In [ ]:
# HAC: silhouette ~ three episode dummies + time + vix_cycle
sub = cvi_df.dropna(subset=["silhouette", "vix_cycle", "trend_time"])
rd = sub["Rebal_date"]
D1 = rd.between("2012-01-01", "2013-12-31").astype(float)
D2 = rd.between("2015-01-01", "2016-12-31").astype(float)
D3 = rd.between("2019-09-01", "2020-12-31").astype(float)
X = np.column_stack(
    [D1.values, D2.values, D3.values, sub["trend_time"].values, sub["vix_cycle"].values]
)
res = hac_ols(sub["silhouette"].values, X)
with open(RESULTS_DIR / "hypothesis_C_regression_hac.txt", "w", encoding="utf-8") as f:
    f.write(res.summary().as_text())
print(res.summary())


In [ ]:
fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(cvi_df["Rebal_date"], cvi_df["silhouette"], marker="o", lw=1)
for name, a, b in EPISODES:
    ax.axvspan(pd.Timestamp(a), pd.Timestamp(b), alpha=0.2, label=name)
ax.set_title("AggBond: silhouette over biannual refits (shaded = narrative episodes)")
ax.set_ylabel("Silhouette")
ax.grid(alpha=0.3)
ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
ax.legend(loc="lower right", fontsize=7)
plt.tight_layout()
plt.savefig(RESULTS_DIR / "hypothesis_C_aggbond_silhouette_episodes.png", dpi=150)
plt.show()
